In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv
from os import environ
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy.engine.base import Engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import CatBoostEncoder
from sklearn.compose import ColumnTransformer
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
    median_absolute_error
)

# Подключение к БД

In [2]:
load_dotenv(find_dotenv())

True

In [3]:
DB_USER = environ['DB_USER']
DB_HOST = environ['DB_HOST']
DB_PORT = environ['DB_PORT']
DB_NAME = environ['DB_NAME']
DB_PASSWORD = environ['DB_PASSWORD']

In [4]:
db_conn_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

In [5]:
conn = create_engine(db_conn_url)

# Загрузка датасета из БД

In [6]:
sql = "select * from buildings_flats"

In [7]:
data = pd.read_sql(sql, conn)

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100928 entries, 0 to 100927
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   building_id        100928 non-null  int64  
 1   flat_id            100928 non-null  int64  
 2   build_year         100928 non-null  int64  
 3   building_type_int  100928 non-null  int64  
 4   latitude           100928 non-null  float64
 5   longitude          100928 non-null  float64
 6   ceiling_height     100928 non-null  float64
 7   flats_count        100928 non-null  int64  
 8   floors_total       100928 non-null  int64  
 9   has_elevator       100928 non-null  bool   
 10  floor              100928 non-null  int64  
 11  kitchen_area       100928 non-null  float64
 12  living_area        100928 non-null  float64
 13  rooms              100928 non-null  int64  
 14  is_apartment       100928 non-null  bool   
 15  studio             100928 non-null  bool   
 16  to

In [9]:
data.to_csv('data.csv')

# Обучение базовой модели

## Разделение данных на обучающую и тестовую выборки

In [10]:
X_tr, X_val, y_tr, y_val = train_test_split(
    data,
    data['price'],
    test_size=0.2,
    random_state=42
) 

## Обработка признаков

Бинарные признаки:

In [11]:
numeric_features = [
    'build_year',
    'latitude',
    'longitude',
    'ceiling_height',
    'flats_count',
    'floors_total',
    'kitchen_area',
    'living_area',
    'rooms',
    'total_area'
]

Числовые признаки:

In [12]:
binary_features = ['has_elevator', 'is_apartment', 'studio']

Категориальные признаки:

In [13]:
categorical_features = ['building_type_int', 'floor']

### Трансформеры

Кодирование бинарных признаков:

In [14]:
binary_transformer = OneHotEncoder(drop='if_binary', sparse_output=False)

Кодирование категориальных признаков

In [15]:
cat_transformer = CatBoostEncoder(cols=categorical_features)

Нормирование числовых признаков:

In [16]:
numeric_transformer = StandardScaler()

Общий трансформер:

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_transformer, binary_features),
        ('cat', cat_transformer, categorical_features),
        ('num', numeric_transformer, numeric_features)
    ],
    remainder='drop'
)

## Модель

In [18]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)

## Пайплайн для трансформации данных и обучения модели

In [19]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

In [20]:
pipeline.fit(X_tr, y_tr)

0:	learn: 4416439.3458810	total: 61ms	remaining: 1m
100:	learn: 2534135.4133622	total: 572ms	remaining: 5.09s
200:	learn: 2448220.5466536	total: 1.04s	remaining: 4.14s
300:	learn: 2404596.0241265	total: 1.5s	remaining: 3.48s
400:	learn: 2371486.9211534	total: 1.97s	remaining: 2.94s
500:	learn: 2348006.4476545	total: 2.42s	remaining: 2.41s
600:	learn: 2326007.7443511	total: 2.89s	remaining: 1.92s
700:	learn: 2308052.5277322	total: 3.36s	remaining: 1.43s
800:	learn: 2292018.3583659	total: 3.82s	remaining: 949ms
900:	learn: 2276945.2138488	total: 4.29s	remaining: 471ms
999:	learn: 2263571.3550243	total: 4.74s	remaining: 0us


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('binary',
                                                  OneHotEncoder(drop='if_binary',
                                                                sparse_output=False),
                                                  ['has_elevator',
                                                   'is_apartment', 'studio']),
                                                 ('cat',
                                                  CatBoostEncoder(cols=['building_type_int',
                                                                        'floor']),
                                                  ['building_type_int',
                                                   'floor']),
                                                 ('num', StandardScaler(),
                                                  ['build_year', 'latitude',
                                                   'longitude',
                                                   'ceiling_height',
                                                   'flats_count',
                                                   'floors_total',
                                                   'kitchen_area',
                                                   'living_area', 'rooms',
                                                   'total_area'])])),
                ('model',
                 <catboost.core.CatBoostRegressor object at 0x758fa4e2a9b0>)])

## Предсказание

In [21]:
y_pred = pipeline.predict(X_val)

Для оценки модели будет использовать:

* MAE для оценки, насколько в среднем ошибается в рублях;
* MAPE показывает среднюю ошибку в процентах;
* R^2 показывает, насколько модель лучше предсказания средним.

In [22]:
mae = mean_absolute_error(y_val, y_pred)
print(f'MAE: {mae:.2f} руб')

MAE: 1863224.51 руб


In [23]:
mape = mean_absolute_percentage_error(y_val, y_pred) * 100
print(f'MAPE: {mape:.2f}%')

MAPE: 16.68%


In [24]:
r2 = r2_score(y_val, y_pred)
print(f'R^2: {r2:.2f}')

R^2: 0.73


# Оценка модели

R^2 = 0.73 — модель объясняет 73% дисперсии цен. Для задачи оценки недвижимости по базовым признакам это хороший результат.

MAPE = 16.68% — средняя относительная ошибка ~17%. Для рынка недвижимости это приемлемо.

MAE = 1.86 млн руб при медиане цены 10.8 млн — ошибка ~17% от типичной цены. Согласуется с MAPE.

**Вывод:** Модель оценивает квартиру в Москве со средней ошибкой ~17% (≈1.9 млн руб). В 73% случаев вариация цен объясняется признаками дома и квартиры. Модель можно использовать как baseline.